# cytozip Python API
## Download example dataset

```shell
pip install pyfigshare
pip uninstall -y ChunkZIP && pip install git+http://github.com/DingWB/cytozip
# or pip install ChunkZIP
figshare download 25374073 -o czip_example_data
```

## View reference .cz file

### view header

In [ ]:
import os,sys
import cytozip
reader=cytozip.Reader("czip_example_data/FC_E17a_3C_1-1-I3-F13.cz")
reader.print_header()

/anvil/projects/x-mcb130189/Wubin/Github/cytozip/cytozip/__init__.py
magic  :  b'CZIP'
version  :  0.0
total_size  :  30630066
message  :  mm10_with_chrL.allc.cz
formats  :  ['B', 'B']
columns  :  ['mc', 'cov']
dimensions  :  ['chrom']
header_size  :  59


### summary chunks & blocks

In [ ]:
reader.summary_chunks(printout=False)
reader.summary_blocks(printout=False)

Every chunk has a dimension (chrom, sample, cell types or the combination of those dimensions)

### query

In [ ]:
for record in reader.query(dimension="chr9",start=3000294,end=3000472,reference="czip_example_data/mm10_with_chrL.allc.cz",printout=False):
    print(record)

Number of records: 0


In [ ]:
for record in reader.query(dimension="chr9",start=3000294,end=3000472,printout=False):
    print(record)

chrom	pos	mc	cov
chr9	3000294	54	63
chr9	3000342	69	85
chr9	3000354	77	82
chr9	3000381	52	64
chr9	3000382	74	87
chr9	3000399	66	67
chr9	3000441	84	138
chr9	3000457	139	162
chr9	3000458	64	74
chr9	3000472	161	183


### Pack .allc.tsv.gz to .cz without coordinates (using reference)

```shell
cytozip allc2cz FC_E17a_3C_1-1-I3-F13.allc.tsv.gz FC_E17a_3C_1-1-I3-F13.cz -r ~/Ref/mm10/annotations/mm10_with_chrL.allc.cz
# Uses numpy-vectorized implementation for fast conversion

# To enable delta encoding for the position column:
cytozip allc2cz FC_E17a_3C_1-1-I3-F13.allc.tsv.gz FC_E17a_3C_1-1-I3-F13.delta.cz -r ~/Ref/mm10/annotations/mm10_with_chrL.allc.cz --delta True
```

In [ ]:
!czip view -I FC_E17a_3C_1-1-I3-F13.cz --show-dim 0 |head

chrom	mc	cov
chr1	0	0
chr1	0	0
chr1	0	0
chr1	0	0
chr1	0	0
chr1	0	0
chr1	0	0
chr1	0	0
chr1	0	0


view FC_E17a_3C_1-1-I3-F13.cz together with reference

In [ ]:
!czip view -I FC_E17a_3C_1-1-I3-F13.cz --show-dim 0 -r ~/Ref/mm10/annotations/mm10_with_chrL.allc.cz |head

chrom	pos	strand	context	mc	cov
chr1	3000003	+	CTG	0	0
chr1	3000005	-	CAG	0	0
chr1	3000009	+	CTA	0	0
chr1	3000016	-	CAA	0	0
chr1	3000018	-	CAC	0	0
chr1	3000019	-	CCA	0	0
chr1	3000023	+	CTT	0	0
chr1	3000027	-	CAA	0	0
chr1	3000029	-	CTC	0	0


### Query .cz using cytozip

#### query allc.tsv.gz using tabix

In [ ]:
!tabix FC_E17a_3C_1-1-I3-F13.allc.tsv.gz chr9 | awk '$5 > 50' |head

chr9	3000294	-	CAT	54	63	1
chr9	3000342	-	CGA	69	85	1
chr9	3000354	-	CGT	77	82	1
chr9	3000381	+	CGT	52	64	1
chr9	3000382	-	CGG	74	87	1
chr9	3000399	+	CGA	66	67	1
chr9	3000441	+	CGT	84	138	1
chr9	3000457	+	CGT	139	162	1
chr9	3000458	-	CGA	64	74	1
chr9	3000472	+	CGT	161	183	1


#### query allc.cz using cytozip

In [ ]:
!# query is Python-API only: reader.query(dimension="chr9", start=3000294, end=3000472, reference="~/Ref/mm10/annotations/mm10_with_chrL.allc.cz") |awk '$5>50'

chrom	pos	strand	context	mc	cov
chr9	3000294	-	CAT	54	63
chr9	3000342	-	CGA	69	85
chr9	3000354	-	CGT	77	82
chr9	3000381	+	CGT	52	64
chr9	3000382	-	CGG	74	87
chr9	3000399	+	CGA	66	67
chr9	3000441	+	CGT	84	138
chr9	3000457	+	CGT	139	162
chr9	3000458	-	CGA	64	74
chr9	3000472	+	CGT	161	183


## Cat multiple .cz files into one .cz file

In [ ]:
!czip catcz -O cat.cz -F Q,c,3s -C pos,strand,context -D chrom --help

INFO: Showing help with the command 'czip catcz -O cat.cz -F Q,c,3s -C pos,strand,context -D chrom -- --help'.

NAME
    czip catcz -O cat.cz -F Q,c,3s -C pos,strand,context -D chrom - Cat multiple .cz files into one .cz file.

SYNOPSIS
    czip catcz -O cat.cz -F Q,c,3s -C pos,strand,context -D chrom <flags>

DESCRIPTION
    Cat multiple .cz files into one .cz file.

FLAGS
    -I, --Input=INPUT
        Type: Optional[]
        Default: None
        Either a str (including *, as input for glob, should be inside the                      double quotation marks if using fire) or a list.
    -d, --dim_order=DIM_ORDER
        Type: Optional[]
        Default: None
        If dim_order=None, Input will be sorted using python sorted. If dim_order is a list, tuple or array of basename.rstrip(.cz), sorted as dim_order. If dim_order is a file path (for example, chrom size path to dim_order chroms or only use selected chroms) will be sorted as the 1st column of the input file path (without header

```shell
czip catcz -O mm10_with_chrL.allc.cz -F Q,c,3s -C pos,strand,context -D chrom -I "cell_type/*.cz" \
            --dim_order ~/Ref/mm10/mm10_ucsc_with_chrL.chrom.sizes --add_dim True --title "cell_id"
```

In this example, we cat multiple .cz file into one .cz file and add another dimension to each chunk (cell_id)

## Extract CG from .cz and merge strand

In [ ]:
!cytozip extractCG --help

INFO: Showing help with the command 'cytozip extractCG -- --help'.

NAME
    cytozip extractCG - Extract CG context from .cz file

SYNOPSIS
    cytozip extractCG <flags>

DESCRIPTION
    Extract CG context from .cz file

FLAGS
    -i, --input=INPUT
        Type: Optional[]
        Default: None
    -o, --outfile=OUTFILE
        Type: Optional[]
        Default: None
    -s, --ssi=SSI
        Type: Optional[]
        Default: None
        ssi should be ssi to mm10_with_chrL.allc.cz.CGN.ssi, not forward strand ssi, but after merge (if merge_cg is True), forward ssi mm10_with_chrL.allc.cz.+CGN.ssi should be used to generate reference, one can
    -c, --chunksize=CHUNKSIZE
        Default: 5000
    -m, --merge_cg=MERGE_CG
        Default: False
        after merging, only forward strand would be kept, reverse strand values would be added to the corresponding forward strand.


```shell
cytozip extractCG -i cz/FC_P13a_3C_2-1-E5-D13.cz -o FC_P13a_3C_2-1-E5-D13.CGN.cz -s ~/Ref/mm10/annotations/mm10_with_chrL.allc.cz.CGN.ssi

# view CG .cz
czip view -I FC_P13a_3C_2-1-E5-D13.CGN.cz --show-dim 0 -r ~/Ref/mm10/annotations/mm10_with_chrL.allCG.forward.cz
```

## Merge multiple .cz files into one .cz file
sum up the mc and cov

In [ ]:
!cytozip merge_cz --help

INFO: Showing help with the command 'cytozip merge_cz -- --help'.

NAME
    cytozip merge_cz - Merge multiple .cz files. For example: cytozip merge_cz -i ./ -o major_type.2D.txt -n 96 -f 2D                           -P ~/Ref/mm10/mm10_ucsc_with_chrL.main.chrom.sizes.txt                           -r ~/Ref/mm10/annotations/mm10_with_chrL.allCG.forward.cz

SYNOPSIS
    cytozip merge_cz <flags>

DESCRIPTION
    Merge multiple .cz files. For example: cytozip merge_cz -i ./ -o major_type.2D.txt -n 96 -f 2D                           -P ~/Ref/mm10/mm10_ucsc_with_chrL.main.chrom.sizes.txt                           -r ~/Ref/mm10/annotations/mm10_with_chrL.allCG.forward.cz

FLAGS
    -i, --indir=INDIR
        Type: Optional[]
        Default: None
        If cz_paths is not provided, indir will be used to get cz_paths.
    --cz_paths=CZ_PATHS
        Type: Optional[]
        Default: None
    --class_table=CLASS_TABLE
        Type: Optional[]
        Default: None
        If class_table is given,

```shell
cytozip merge_mz -i cz-CGN/ -o merged.cz
```

## Merge .cz files belonging to the same cell type
sum up the mc and cov

In [ ]:
!cytozip merge_cell_type --help

INFO: Showing help with the command 'cytozip merge_cell_type -- --help'.

NAME
    cytozip merge_cell_type

SYNOPSIS
    cytozip merge_cell_type <flags>

FLAGS
    -i, --indir=INDIR
        Type: Optional[]
        Default: None
    -c, --cell_table=CELL_TABLE
        Type: Optional[]
        Default: None
    -o, --outdir=OUTDIR
        Type: Optional[]
        Default: None
    -n, --n_jobs=N_JOBS
        Default: 64
    -P, --Path_to_chrom=PATH_TO_CHROM
        Type: Optional[]
        Default: None
    -e, --ext=EXT
        Default: '.CGN.merged.cz'


## Run cytozip allc2cz on GCP

```shell
wget https://raw.githubusercontent.com/DingWB/cytozip/main/data/allc2mz.smk
```

```shell
snakemake --printshellcmds --immediate-submit --notemp -s allc2mz.smk --config indir="gs://mouse_pfc/test_allc" outdir="test_mz" \
            reference="mm10_with_chrL.allc.cz" ref_prefix="gs://wubin_ref/mm10/annotations" \
            chrom="mm10_ucsc_with_chrL.main.chrom.sizes.txt" chrom_prefix="gs://wubin_ref/mm10" \
            gcp=True --default-remote-prefix mouse_pfc --default-remote-provider GS \
            --google-lifesciences-region us-west1 --scheduler greedy -j 96 -np
```

## Run cytozip extractCG on GCP

```shell
wget https://raw.githubusercontent.com/DingWB/cytozip/main/data/extractCG.smk
```

```shell
snakemake --use-conda --printshellcmds -s extractCG.smk \
          --config algorithm="bmzip" indir=test_mz files_path=mz.path".0$SKYPILOT_NODE_RANK" \
          outdir=pfc_mz-CGN bmi=mm10_with_chrL.allc.cz.CGN.bmi bmi_prefix=gs://wubin_ref/mm10/annotations \
          gcp=True --default-remote-prefix mouse_pfc --default-remote-provider GS \
          --google-lifesciences-region us-west1 --scheduler greedy -j 96
```

## Converting .cz file back to allc.tsv.gz

convert both CG and CH to allc

In [ ]:
!czip view -I FC_E17a_3C_1-1-I3-F13.cz --show-dim 0 -r ~/Ref/mm10/annotations/mm10_with_chrL.allc.cz --no-header |head

chr1	3000003	+	CTG	0	0
chr1	3000005	-	CAG	0	0
chr1	3000009	+	CTA	0	0
chr1	3000016	-	CAA	0	0
chr1	3000018	-	CAC	0	0
chr1	3000019	-	CCA	0	0
chr1	3000023	+	CTT	0	0
chr1	3000027	-	CAA	0	0
chr1	3000029	-	CTC	0	0
chr1	3000030	-	CCT	0	0


```shell
czip view -I FC_E17a_3C_1-1-I3-F13.cz --show-dim 0 -r ~/Ref/mm10/annotations/mm10_with_chrL.allc.cz --no-header | 
    awk 'BEGIN{FS=OFS="\t"}; {print $0,1}' | bgzip > test1.allc.tsv.gz && tabix -f -s 1 -b 2 -e 2 test1.allc.tsv.gz
```

convert only CG to allc

In [ ]:
!czip view -I FC_E17a_3C_1-1-I3-F13.cz --show-dim 0 -r ~/Ref/mm10/annotations/mm10_with_chrL.allCG.forward.cz --no-header |head

chr1	3000827	+	CGT	0	0
chr1	3001007	+	CGG	0	0
chr1	3001018	+	CGT	0	0
chr1	3001277	+	CGA	0	0
chr1	3001629	+	CGT	0	0
chr1	3003226	+	CGG	0	0
chr1	3003339	+	CGC	0	0
chr1	3003379	+	CGT	0	0
chr1	3003582	+	CGC	0	0
chr1	3003640	+	CGG	0	0


```shell
czip view -I FC_E17a_3C_1-1-I3-F13.cz --show-dim 0 -r ~/Ref/mm10/annotations/mm10_with_chrL.allCG.forward.cz --no-header | 
    awk 'BEGIN{FS=OFS=\"\t\"}; {print \$0,1}' | bgzip > test1.CG.allc.tsv.gz && tabix -f -s 1 -b 2 -e 2 test1.CG.allc.tsv.gz
```

## Write and Read a .cz File

### Writer example

In [ ]:
import struct, tempfile, os
import cytozip

# Create a .cz file
tmpfile = tempfile.mktemp(suffix=".cz")
w = cytozip.Writer(
    output=tmpfile,
    formats=["Q", "H", "H"],
    columns=["pos", "mc", "cov"],
    dimensions=["chrom"],
)

# Write some records
records = [(100, 1, 10), (200, 2, 15), (350, 3, 20), (500, 0, 8)]
data = b"".join(struct.pack("<QHH", *r) for r in records)
w.write_chunk(data, ["chr1"])
w.close()
print(f"File written to: {tmpfile}")

2026-04-05 23:43:53.210 | DEBUG    | cytozip.cz:<module>:84 - Cython accelerated functions loaded successfully


File written to: /tmp/tmprvyawzhv.cz


In [2]:
# Read back
r = cytozip.Reader(tmpfile)
print(f"Magic: {r.header['magic']}")
print()
for record in r.fetch(("chr1",)):
    print(record)
r.close()
os.unlink(tmpfile)

Magic: b'CZIP'

[100, 1, 10]
[200, 2, 15]
[350, 3, 20]
[500, 0, 8]
[100, 1, 10]
[200, 2, 15]
[350, 3, 20]
[500, 0, 8]


### allc2cz conversion

In [ ]:
# Python API for allc2cz
# cytozip.allc2cz(
#     input="FC_E17a_3C_1-1-I3-F13.allc.tsv.gz",
#     outfile="FC_E17a_3C_1-1-I3-F13.cz",
#     reference="~/Ref/mm10/annotations/mm10_with_chrL.allc.cz",
# )

# AllC class for generating reference .cz files
# allc = cytozip.AllC(genome="hg38.fa", output="hg38_allc.cz")
# allc.run()
print("allc2cz converts .allc.tsv.gz to .cz format")

allc2cz converts .allc.tsv.gz to .cz format


## Remote Reading (from URL)
cytozip files with a chunk index can be read directly from HTTP/HTTPS URLs, without downloading the entire file. This uses HTTP Range requests to fetch only the required chunks.

Key features:
- **RemoteFile**: File-like object backed by HTTP Range requests with 2MB read-ahead cache
- **Reader.from_url()**: Open a remote .cz file with O(1) chunk lookup via chunk index
- **Auto-detection**: Passing a URL to `Reader()` automatically uses remote mode

In [ ]:
import cytozip
from cytozip.cz import RemoteFile

# Example: open a remote .cz file
# url = "https://your-server.com/data/sample.cz"

# Method 1: Using Reader.from_url (recommended)
# reader = cytozip.Reader.from_url(url)

# Method 2: Pass URL directly to Reader (auto-detected)
# reader = cytozip.Reader(url)

# Once opened, all Reader methods work the same as local files:
# reader.print_header()
# reader.summary_chunks()
# for record in reader.fetch(("chr1",)):
#     print(record)
# for record in reader.query(dimension="chr1", start=1000, end=2000, printout=False):
#     print(record)
# reader.close()

print("Remote reading requires an HTTP server supporting Range requests.")
print("The .cz file should include a chunk index for optimal performance.")

### Low-level RemoteFile usage
RemoteFile can also be used standalone as a file-like object for custom HTTP Range-based reading:

In [ ]:
from cytozip.cz import RemoteFile

# RemoteFile provides a file-like interface over HTTP Range requests
# rf = RemoteFile(url, cache_size=2*1024*1024)  # 2MB read-ahead cache
# print(f"Remote file size: {rf.size} bytes")
# 
# # Supports standard file operations
# data = rf.read(100)      # read 100 bytes
# rf.seek(0)               # seek to beginning
# rf.seek(-100, 2)         # seek 100 bytes from end
# pos = rf.tell()          # current position
# rf.close()

print("RemoteFile(url, cache_size=2*1024*1024)")
print("  .read(size)  - read bytes (with read-ahead caching)")
print("  .seek(offset, whence)  - seek to position")
print("  .tell()  - return current position")
print("  .size  - total file size (from Content-Length header)")